In [ ]:
# Cell 1: Imports
import os
from huggingface_hub import login
from datasets import load_dataset

from src.model_loader import load_model
from src.tasks import CaptionTask, ZeroShotTask
from src.datasets_adapter import from_derm1m, from_image_folder
from src.runner import explain

login(token=os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
# Cell 2: Define model zoo
MODELS = [
    # {"name": "hf-hub:redlessone/DermLIP_ViT-B-16", "backend": "open_clip"},
    {"name": "coca_ViT-B-32", "backend": "open_clip",
     "pretrained": "laion2b_s13b_b90k"},
    {"name": "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224",
     "backend": "open_clip",
     "hf_tokenizer_name": "microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"},
]

In [12]:
# Cell 3: TASK A — Caption-based explanation on Derm1M
ds = load_dataset("redlessone/Derm1M")

derm1m_entries = [
    {"filename": "pubmed/0d_59_PMC4458964_IJD_60_321e_g003_0.png", "index": 132556},
    {"filename": "IIYI/2281_1.png", "index": 126},
]
caption_samples = from_derm1m(ds, derm1m_entries, image_root="derm_train_images")
caption_task = CaptionTask(samples=caption_samples)

In [13]:
# Cell 4: TASK B — Zero-shot classification on HAM10000
HAM7 = [
    "melanocytic nevus", "melanoma", "basal cell carcinoma",
    "benign keratosis", "actinic keratosis",
    "vascular lesion", "dermatofibroma",
]
zs_samples = from_image_folder([
    "ham_images/sample_2_melanoma.jpg",
    # "ham_images/sample_3_melanoma.jpg",
])
zeroshot_task = ZeroShotTask(
    samples=zs_samples,
    class_names=HAM7,
    prompt_template=lambda c: f"This image shows a case of {c}",
    explain_classes="top1",  # or "all" / "topk"
    top_k=3,
)

In [ ]:
# Cell 5: Run everything
all_results = {}
for cfg in MODELS:
    model = load_model(**cfg)
    all_results[model.name] = {
        "caption": explain(model, caption_task, budget=2**10),
        "zeroshot": explain(model, zeroshot_task, budget=2**8),
    }
    del model
    import torch; torch.cuda.empty_cache()